# Course 1: From Beginner to an AI Research Lab
## Session 5: Deep learning from scratch

**Goal:** build a tiny automatic differentiation engine, train a network with it, then run the two checks that catch most training bugs.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. The `Value` class
Each `Value` remembers the values it was computed from and how to pass a gradient back to them. Read each `_backward` until you can explain it.

In [ ]:
import math

class Value:
    def __init__(self, data, parents=()):
        self.data, self.grad = data, 0.0
        self._parents, self._backward = parents, lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,))
        def _backward():
            self.grad += (1 - t * t) * out.grad
        out._backward = _backward
        return out

    __radd__ = __add__
    __rmul__ = __mul__
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other if isinstance(other, Value) else -other)

    def backward(self):
        order, seen = [], set()
        def visit(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

a, b = Value(2.0), Value(-3.0)
out = (a * b + a).tanh()
out.backward()
print(a.grad, b.grad)

### 2. Check against PyTorch

In [ ]:
import torch
ta, tb = torch.tensor(2.0, requires_grad=True), torch.tensor(-3.0, requires_grad=True)
torch.tanh(ta * tb + ta).backward()
print("PyTorch:", ta.grad.item(), tb.grad.item())
assert abs(a.grad - ta.grad.item()) < 1e-6 and abs(b.grad - tb.grad.item()) < 1e-6
assert abs(a.grad + 0.00268) < 1e-5 and abs(b.grad - 0.00268) < 1e-5

### 3. A neuron, a layer, a network, trained on two moons
Plain gradient descent, using only `Value` objects. It is slow, and that is fine: the point is that nothing is hidden.

In [ ]:
import random
from sklearn.datasets import make_moons
random.seed(0)
class Neuron:
    def __init__(self, n_in): self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]; self.b = Value(0.0)
    def __call__(self, x): return sum((wi * xi for wi, xi in zip(self.w, x)), self.b).tanh()
    def parameters(self): return self.w + [self.b]
class Layer:
    def __init__(self, n_in, n_out): self.neurons = [Neuron(n_in) for _ in range(n_out)]
    def __call__(self, x): return [n(x) for n in self.neurons]
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]
class MLP:
    def __init__(self, sizes): self.layers = [Layer(a, b) for a, b in zip(sizes, sizes[1:])]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x[0]
    def parameters(self): return [p for l in self.layers for p in l.parameters()]

Xm, ym = make_moons(n_samples=60, noise=0.15, random_state=0); ym = 2 * ym - 1          # labels -1 and +1
net = MLP([2, 8, 1])
def run_epoch(lr):
    preds = [net(list(x)) for x in Xm]
    loss = sum(((p - int(t)) * (p - int(t)) for p, t in zip(preds, ym)), Value(0.0)) * (1 / len(ym))
    for p in net.parameters(): p.grad = 0.0
    loss.backward()
    for p in net.parameters(): p.data -= lr * p.grad
    return loss.data, sum((p.data > 0) == (t > 0) for p, t in zip(preds, ym)) / len(ym)
for epoch in range(120):
    l, acc = run_epoch(0.3)
    if epoch % 30 == 0 or epoch == 119: print(f"epoch {epoch:3d}   loss {l:.3f}   accuracy {acc:.2f}")
assert acc > 0.85

### 4. The same kind of network in PyTorch, and two checks before training
MNIST needs a download, so the runnable cells use scikit-learn's 8x8 digits (64 inputs, 10 classes). The checks are the same for any classifier:
1. **Loss at initialization.** With 10 classes and no knowledge, the loss should be near `ln(10) = 2.30`.
2. **Overfit one small batch.** If the network cannot drive the loss to almost zero on 8 examples, there is a bug.

In [ ]:
import torch.nn as nn, torch.nn.functional as F, numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.manual_seed(0)
d = load_digits(); Xd = torch.tensor(d.data / 16.0, dtype=torch.float32); yd = torch.tensor(d.target)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=0, stratify=yd)
def make(): return nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 10))

model = make()
init_loss = F.cross_entropy(model(Xtr), ytr).item()
print(f"check 1: loss at initialization {init_loss:.3f} (expected about {np.log(10):.3f})")
assert abs(init_loss - np.log(10)) < 0.15

opt = torch.optim.Adam(model.parameters(), lr=0.001); xb, yb = Xtr[:8], ytr[:8]
for step in range(400):
    loss = F.cross_entropy(model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
print(f"check 2: loss on one batch of 8 after 400 steps {loss.item():.5f}")
assert loss.item() < 0.01

### 5. A bug that the checks catch
A classic: applying softmax before `cross_entropy`, which applies it again. Training still 'works', only worse. The one-batch check exposes it.

In [ ]:
torch.manual_seed(0); buggy = make(); opt = torch.optim.Adam(buggy.parameters(), lr=0.001)
for step in range(400):
    loss = F.cross_entropy(torch.softmax(buggy(xb), dim=1), yb); opt.zero_grad(); loss.backward(); opt.step()    # BUG: softmax twice
print(f"with the bug, loss on the same batch after 400 steps: {loss.item():.3f} (it cannot get near zero)")
assert loss.item() > 1.0

### 6. Train and report

In [ ]:
torch.manual_seed(0); model = make(); opt = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(30):
    for idx in torch.randperm(len(Xtr)).split(32):
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx]); opt.zero_grad(); loss.backward(); opt.step()
acc = (model(Xte).argmax(1) == yte).float().mean().item()
print(f"test accuracy on the 8x8 digits: {acc:.3f}")
assert acc > 0.93

### 7. MNIST (NOT RUN here)

In [ ]:
# NOT RUN: needs torchvision and a download of MNIST.
import torchvision, torchvision.transforms as T
train = torchvision.datasets.MNIST("data", train=True, download=True, transform=T.ToTensor())
test = torchvision.datasets.MNIST("data", train=False, download=True, transform=T.ToTensor())
loader = torch.utils.data.DataLoader(train, batch_size=64, shuffle=True)
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)); opt = torch.optim.Adam(model.parameters(), lr=0.001)
# 1. Run check 1 (expect about 2.30) and check 2 (one batch of 8, loss below 0.001 within about 200 steps).
# 2. Train for 3 epochs and report test accuracy. A network of this size normally reaches about 97%; report what you get.

### TODO
1. Add `relu` and `exp` to `Value`, with tests against PyTorch.
2. In section 3, forget to zero the gradients. What happens, and why?
3. Run the MNIST cell with both checks and hand in the three numbers.